In [111]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import pickle
import os 
import warnings
warnings.filterwarnings('ignore')

In [112]:
dataset = pd.read_csv("../data/amazon_products.csv", nrows=10000)
print(f"===Dataset Loaded===")
print(f"Columns: {dataset.columns.to_list()}")

===Dataset Loaded===
Columns: ['asin', 'title', 'imgUrl', 'productURL', 'stars', 'reviews', 'price', 'listPrice', 'category_id', 'isBestSeller', 'boughtInLastMonth']


In [113]:
# Loading Content Model
from sklearn.metrics.pairwise import cosine_similarity
try:
    with open('../artifacts/tfidf_vectorizer.pkl', 'rb') as f:
        vectorizer = pickle.load(f)
    print("===Loaded TF-IDF vectorizer===")

except:
    print("Vectorizer Not found Creating new one...")
    vectorizer = TfidfVectorizer(stop_words='english', max_features=2000)

# Create product features
def create_text(row):
    features = [str(row['title']), f"category_{row['category_id']}"]
    if row['price'] > 0:
        if row['price'] < 50:
            features.append("Budget")
        elif row['price'] < 200:
            features.append("Mid-range")
        else:
            features.append("Premium")
    
    if row['isBestSeller']:
        features.append("bestseller")
    return " ".join(features)

dataset['product_text'] = dataset.apply(create_text, axis = 1)

# Create TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(dataset['product_text'])

print(f" TF-IDF matrix shape: {tfidf_matrix.shape}")
print("Contel Model is ready - using Cosine Similarity")


===Loaded TF-IDF vectorizer===
 TF-IDF matrix shape: (10000, 5000)
Contel Model is ready - using Cosine Similarity


In [114]:
# Calculating popularity scores 
max_bought = dataset['boughtInLastMonth'].max()
dataset['popularity_score'] = dataset['boughtInLastMonth'] / max_bought if max_bought > 0 else 0

print(" Popularity scores calculated")
print(f"   Range: {dataset['popularity_score'].min():.2f} to {dataset['popularity_score'].max():.2f}")

 Popularity scores calculated
   Range: 0.00 to 1.00


In [115]:
# Create content function using cosine similarity
def get_content_similarity(product_idx):
    """Get content-based similarity scores using cosine_similarity"""
    content_scores = cosine_similarity(tfidf_matrix[product_idx], tfidf_matrix)[0]
    # Get top 51 indices (including itself)
    indices = content_scores.argsort()[::-1][:51]
    scores = content_scores[indices]
    return indices, scores
def recommend_content(dataset, product_name, n=5):
    """Pure content recommendation"""
    matching = dataset[dataset['title'].str.contains(product_name, case=False)]
    if len(matching) == 0:
        return f"Product '{product_name}' not found"
    
    product_idx = matching.index[0]
    indices, scores = get_content_similarity(product_idx)
    
    recommendations = []
    for i in range(1, min(len(indices), n+1)):
        idx = indices[i]
        recommendations.append({
            'title': dataset.iloc[idx]['title'],
            'price': dataset.iloc[idx]['price'],
            'stars': dataset.iloc[idx]['stars'],
            'similarity': round(scores[i], 3)
        })
    
    return pd.DataFrame(recommendations[:n])

print(" Content Model ready - using cosine_similarity)")
print(f"Test content recommendation: {recommend_content(dataset, 'Socks', 3).iloc[0]['title'][:40] if len(recommend_content(dataset, 'Socks', 3)) > 0 else 'N/A'}...")

 Content Model ready - using cosine_similarity)
Test content recommendation: Performance Lightweight Crew Training So...


In [116]:
# Hybrid Recommender- combines both - Popularity recommender and content based recommender
def hybrid_recommend(product_name, n=5, content_weight=0.7, popularity_weight=0.3):
    """Combine content + popularity scores"""
    
    matching = dataset[dataset['title'].str.contains(product_name, case=False)]
    if len(matching) == 0:
        return f"Product '{product_name}' not found"
    
    product_idx = matching.index[0]
    
    # Get content scores
    content_scores = cosine_similarity(tfidf_matrix[product_idx], tfidf_matrix)[0]
    
    # Calculate hybrid scores for all products
    recommendations = []
    for idx in range(len(dataset)):
        if idx != product_idx:
            content_score = content_scores[idx]
            popularity_score = dataset.iloc[idx]['popularity_score']
            hybrid_score = (content_weight * content_score) + (popularity_weight * popularity_score)
            
            recommendations.append({
                'title': dataset.iloc[idx]['title'],
                'price': dataset.iloc[idx]['price'],
                'stars': dataset.iloc[idx]['stars'],
                'bought': dataset.iloc[idx]['boughtInLastMonth'],
                'content_score': round(content_score, 3),
                'popularity_score': round(popularity_score, 3),
                'hybrid_score': round(hybrid_score, 3)
            })
    
    recommendations.sort(key=lambda x: x['hybrid_score'], reverse=True)
    return pd.DataFrame(recommendations[:n])

print(" Hybrid Recommender ready!")

 Hybrid Recommender ready!


In [117]:
# Testing  Hybrid Recommender model
print("="*60)
print(" TEST: Recommendations for 'Socks'")
print("="*60)

results = hybrid_recommend("Socks", 5)

if isinstance(results, pd.DataFrame):
    for i, row in results.iterrows():
        print(f"\n{i+1}. {row['title'][:60]}...")
        print(f"  ${row['price']} |  {row['stars']}")
        print(f"  Hybrid Score: {row['hybrid_score']}")
        print(f"  (Content: {row['content_score']} + Popularity: {row['popularity_score']})")
else:
    print(results)

 TEST: Recommendations for 'Socks'

1. Performance Lightweight Crew Training Socks (3 Pair)...
  $20.0 |  4.4
  Hybrid Score: 0.553
  (Content: 0.788 + Popularity: 0.005)

2. Dri-Fit Training Cotton Cushioned Crew Socks 6 PAIR Black wi...
  $24.0 |  4.6
  Hybrid Score: 0.461
  (Content: 0.444 + Popularity: 0.5)

3. mens Performance Cushion Crew Training Socks (3 Pairs)...
  $14.5 |  4.6
  Hybrid Score: 0.436
  (Content: 0.494 + Popularity: 0.3)

4. Men's Dri-Fit Training Cotton Cushioned Crew Socks (6 Pair) ...
  $29.0 |  4.2
  Hybrid Score: 0.415
  (Content: 0.584 + Popularity: 0.02)

5. Men/'s Performance Cotton Cushioned Crew Socks, 6 Pair Mediu...
  $39.95 |  4.3
  Hybrid Score: 0.409
  (Content: 0.585 + Popularity: 0.0)


In [118]:
# Comparing  all 3 models
print("\n" + "="*60)
print(" COMPARING ALL MODELS")
print("="*60)

test_product = "Phone"

# Popularity model
print("\n Popularity Only:")
popular_results = dataset.nlargest(5, 'popularity_score')[['title', 'price']]
for i, row in popular_results.iterrows():
    print(f"   {i+1}. {row['title'][:50]}... (${row['price']})")

# Content_based model
print("\n Content_based Only:")
content_results = recommend_content(dataset,test_product, 5)
if isinstance(content_results, pd.DataFrame):
    for i, row in content_results.iterrows():
        print(f"   {i+1}. {row['title'][:50]}... (${row['price']})")

# Hybrid model
print("\n Hybrid:")
hybrid_results = hybrid_recommend(test_product, 5)
if isinstance(hybrid_results, pd.DataFrame):
    for i, row in hybrid_results.iterrows():
        print(f"   {i+1}. {row['title'][:50]}... (${row['price']})")


 COMPARING ALL MODELS

 Popularity Only:
   925. Men's Eversoft Cotton Stay Tucked Crew T-Shirt... ($18.48)
   947. mens 12 Pair Pack Dual Defense Cushioned Casual So... ($15.49)
   928. Men's Boxer Briefs, Soft and Breathable Cotton Und... ($21.98)
   934. Men's Crew T-Shirts, Multipack, Style G1100... ($18.99)
   935. Nxtrnd XTD Scrunch Football Socks, Extra Long Padd... ($19.95)

 Content_based Only:
   1. Travel Painting Travel Luggage Cover Suitcase Prot... ($36.49)
   2. Carry On Luggage 20'' Travel Suitcase Rolling Lugg... ($169.99)
   3. Luggage Suitcase with Spinner Wheels, 24'' Checked... ($199.99)
   4. Large Luggage Suitcase with Wheels, Check-in 24 In... ($129.99)
   5. Sports Baseball Sliding Shorts with Cup Holder - Y... ($16.99)

 Hybrid:
   1. Men's Eversoft Cotton Stay Tucked Crew T-Shirt... ($18.48)
   2. mens 12 Pair Pack Dual Defense Cushioned Casual So... ($15.49)
   3. Men's Boxer Briefs, Soft and Breathable Cotton Und... ($21.98)
   4. Men's Crew T-Shirts, Mult